# Analisis EDA: OnLine Retail II

## 1. Carga de Datos

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.options.display.float_format = '{:,.2f}'.format

df=pd.read_csv("../data/online_retail_II.csv")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


> **Hallazgo:** dataset cargado con 8 columnas. Cada fila es un producto vendido dentro de una factura.

## 2. Estructura del dataset

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


> **Hallazgo:** 1.067.371 filas. `Customer ID` tiene 243.007 nulos (~22%) y `Description` 4.382 nulos.

## 3. Filtrado de clientes sin ID

In [23]:
df_clean = df.dropna(subset=['Customer ID'])
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)
df_clean.shape

(824364, 8)

> **Hallazgo:** quedan 824.364 filas. Se descartan las filas sin cliente porque no permiten analizar comportamiento de compra.

## 4. Valores negativos

In [24]:
print(df_clean[df_clean['Quantity'] < 0].shape)
print(df_clean[df_clean['Price'] < 0].shape)

(18744, 8)
(0, 8)


> **Hallazgo:** 18.744 filas con `Quantity` negativa. No hay precios negativos.

## 5. Cancelaciones

In [25]:
df_clean[df_clean['Quantity'] < 0].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321,Australia


> **Hallazgo:** las facturas con cantidad negativa empiezan con "C", son devoluciones. Se analizarán por separado de las ventas.

## 6. Duplicados y formato de fecha

In [ ]:
duplicados = df_clean.duplicated().sum()
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
print(duplicados)
df_clean.dtypes

> **Hallazgo:** 26.479 filas duplicadas. `InvoiceDate` convertida a datetime.

## 7. Inspección de duplicados

In [ ]:
dups = df_clean[df_clean.duplicated(keep=False)]
dups.sort_values(['Invoice','StockCode']).head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329,United Kingdom


> **Hallazgo:** son líneas idénticas dentro de la misma factura. Se eliminarán para no inflar cantidades ni ingresos.

## 8. Eliminación de duplicados

In [ ]:
df_clean = df_clean.drop_duplicates()
df_clean.shape

(797885, 8)

> **Hallazgo:** Se eliminaron los duplicados y quedaron 797.885 filas

## 9. Estadisticas descriptivas

In [ ]:
df_clean[['Quantity', 'Price']].describe()

,Quantity,Price
count,"797,885.00","797,885.00"
mean,12.60,3.70
std,191.67,71.39
min,"-80,995.00",0.00
25%,2.00,1.25
50%,5.00,1.95
75%,12.00,3.75
max,"80,995.00","38,970.00"


> **Hallazgo:**
> - **Precios en 0:** existen productos con `Price` = 0. Pueden ser regalos, muestras o errores; se revisarán.
> - **Outliers:** el 75% de las filas tiene hasta 12 unidades y £3.75, pero los máximos llegan a 80.995 unidades y £38.970.
> - **Asimetría:** el promedio duplica a la mediana (`Quantity` 12.6 vs 5; `Price` 3.70 vs 1.95). La mayoría de las compras son chicas y pocas compras grandes inflan el promedio. Se usará la **mediana** como valor típico.

## 10. Verificación de outlier en Quantity

In [ ]:
df_clean[df_clean['Quantity'].abs() == 80995]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446,United Kingdom
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446,United Kingdom


> **Hallazgo:**
> - Compra: `Invoice` 581483 , 80995 unidades de `StockCode` 23843 ,`Description` PAPER CRAFT, LITTLE BIRDIE, `Price` £2,08 a las 09:15 HS
> - Devolución: `Invoice` C581484  , mismas unidades, a las 09:27 HS
> - Conclusión: Rechazaron el pedido 12 minutos después de la compra,  aparentemente "error de carga"
> - Decisión: Se eliminan ambas filas para que podamos seguir haciendo el análisis ya que se entiende como un error humano

## 11. Eliminación del outlier de Quantity

In [ ]:
df_clean = df_clean.drop([1065882, 1065883])
df_clean.shape

(797883, 8)

> **Hallazgo:**
> - Se eliminó la compra `Invoice` 581483 y `Invoice` C581484 ya que se presume que fue un fallo humano
> - Quedaron 797.883 filas

## 12. Productos con precio 0

In [ ]:
df_clean[df_clean['Price']==0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.00,16126,United Kingdom
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.00,15658,United Kingdom
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.00,17231,United Kingdom
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.00,14108,United Kingdom
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.00,14108,United Kingdom
...,...,...,...,...,...,...,...,...
1004540,577129,22464,HANGING METAL HEART LANTERN,4,2011-11-17 19:52:00,0.00,15602,United Kingdom
1005014,577168,M,Manual,1,2011-11-18 10:42:00,0.00,12603,Germany
1006110,577314,23407,SET OF 2 TRAYS HOME SWEET HOME,2,2011-11-18 13:23:00,0.00,12444,Norway
1011446,577696,M,Manual,1,2011-11-21 11:57:00,0.00,16406,United Kingdom


> **Hallazgo:**
> - Se hallaron 70 filas con `Price` 0 de 797.883 filas que es el 0,01% del total y algunas con `StockCode` M se entiende que es un cambio manual y algunos son regalos, no se tendran en cuenta para el resto del analisis

## 13.  Eliminación de productos con precios 0

In [ ]:
df_clean = df_clean[df_clean['Price'].abs() > 0]
df_clean.shape

(797813, 8)

> **Hallazgo:**
> - Se eliminarion filas con `Price` igual a 0 y quedaron 797.813 filas

## 14. Separación de ventas y devoluciones

In [ ]:
ventas = df_clean[df_clean['Quantity'] > 0]
devoluciones = df_clean[df_clean['Quantity'] < 0]
print("Devoluciones", devoluciones.shape)
print("Ventas", ventas.shape)
print("Total", ventas.shape[0] + devoluciones.shape[0])

Devoluciones (18389, 8)
Ventas (779424, 8)
Total 797813


> **Hallazgo:**
> - Devoluciones: tiene 18.389 filas
> - Ventas: tiene 779.424 filas 
> - Comprobación: la suma de ventas y devoluciones da 797.813, igual que `df_clean`. No hay filas con `Quantity` igual a 0.
> - Interpretación: las devoluciones son solo el 2,30% del total, un porcentaje bajo. El análisis de ingresos se hará sobre la tabla ventas.

## 15. Ingresos por linea de ventas

In [ ]:
ventas['TotalPrice'] = ventas['Quantity'] * ventas['Price']
ventas['TotalPrice'].describe()


count   779,424.00
mean         22.08
std         123.77
min           0.00
25%           4.95
50%          12.48
75%          19.80
max      77,183.60
Name: TotalPrice, dtype: float64

> **Hallazgo:**
> - **Asimetria:** Se encontro una media de 22,08 que duplica a la mediana de 12,48 y con un minimo de 0,001 y un maximo de 77.183,60 que es sospechoso y se revisara
> - El 75% de las lineas de ventas no superan los £19,80

## 16. Verificación de outlier en TotalPrice

In [ ]:
ventas[ventas['TotalPrice'] == 77183.6].style.format({'Price': '£{:,.2f}', 'TotalPrice': '£{:,.2f}'})


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,£1.04,12346,United Kingdom,"£77,183.60"


In [ ]:
devoluciones[(devoluciones['StockCode'] == '23166') & (devoluciones['Customer ID'] == 12346)].style.format({'Price': '£{:,.2f}'})

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,£1.04,12346,United Kingdom


> **Hallazgo:**
> - **Venta:** `Invoice` 541431, `Quantity` 74215, `StockCode` 23166, `Price` 1.04, `InvoiceDate` 2011-01-18 10:01HS
> - **Devolución:** `Invoice` C541433, misma cantidad, `InvoiceDate` 2011-01-18 10:17HS
> - **Conclusión:**  La devolución se hizo a los 16 minutos posteriores, esto da a entender que es la devolución de la venta anteriormente visualizada 
> - **Decisión:** Se eliminar{a} de ventas y de devoluciones ya que se presume que fue un fallo humano

## 17. Eliminación del outlier de TotalPrice

In [ ]:
ventas = ventas.drop(587080)
ventas.shape

(779423, 9)

In [ ]:
devoluciones = devoluciones.drop(587085)
devoluciones.shape

(18388, 8)

> **Hallazgo:**
> - Se eliminaron las filas 587.080 de ventas y la fila 587.085 de devoluciones
> - **Ventas:** quedo con 779.423 filas
> - **Devoluciones:** quedo con 18.388 filas

## 18. KPI generales de ventas

In [ ]:
ingresos_totales = ventas['TotalPrice'].sum()
print(f"Los ingresos totales son: {ingresos_totales}")

Los ingresos totales son: 17129151.067999996
